In [2]:
import pandas as pd
import requests
import time

In [ ]:
# === CONFIGURACIÓN ===
API_KEY = ''  # 🔑 Reemplaza con tu clave de Google Maps
RADIUS = 3000           # Radio en metros (máx. 50,000)
place_types = ["university", "school"] 

In [24]:
# === 1. LEER COORDENADAS DESDE EXCEL ===
# El Excel debe tener columnas llamadas 'latitud' y 'longitud'
df_coords = pd.read_excel("Centroides de Distritos a Utilizar.xlsx", sheet_name="Centroides y Cuadricula por Dis")

In [27]:
df_coords = df_coords[df_coords['Flag Usar']== 'SI']
df_coords.shape

(1209, 18)

In [28]:
# === 2. FUNCIÓN PARA CONSULTAR LA API ===
# Función para consultar la API (nuevo formato)
def search_places(lat, lon, place_type, radius=3000):
    url = "https://places.googleapis.com/v1/places:searchNearby"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,places.types"
    }

    body = {
        "includedTypes": [place_type],
        "maxResultCount": 10,
        "locationRestriction": {
            "circle": {
                "center": {"latitude": lat, "longitude": lon},
                "radius": radius
            }
        }
    }   

    res = requests.post(url, headers=headers, json=body)
    data = res.json()
    return data

In [36]:
# Recolectar resultados
results = []

for _, row in df_coords.iterrows():
    lat = row["lat"]
    lon = row["lon"]
    for place_type in place_types:
        data = search_places(lat, lon, place_type)
        if "places" in data:
            for p in data["places"]:
                results.append({
                    "latitud": lat,
                    "longitud": lon,
                    "tipo": place_type,
                    "nombre": p.get("displayName", {}).get("text", ""),
                    "direccion": p.get("formattedAddress", ""),
                    "lat": p.get("location", {}).get("latitude", None),
                    "lon": p.get("location", {}).get("longitude", None)
                })
        else:
            print(f"⚠️ Sin resultados para {place_type} en ({lat}, {lon})")
    time.sleep(1)  # para no saturar el límite de solicitudes

# Guardar resultados
pd.DataFrame(results).to_excel("universidades_escuelas_resultados.xlsx", index=False)
print("✅ Búsqueda completada. Resultados guardados en lugares_resultados.xlsx")

⚠️ Sin resultados para university en (13.82591739, -90.1070016)
⚠️ Sin resultados para university en (13.80091739, -90.1070016)
⚠️ Sin resultados para university en (13.85091739, -90.0820016)
⚠️ Sin resultados para university en (13.82591739, -90.0820016)
⚠️ Sin resultados para university en (13.80091739, -90.0820016)
⚠️ Sin resultados para university en (13.77591739, -90.0820016)
⚠️ Sin resultados para university en (13.75091739, -90.0820016)
⚠️ Sin resultados para university en (13.72591739, -90.0820016)
⚠️ Sin resultados para university en (13.87591739, -90.0570016)
⚠️ Sin resultados para university en (13.85091739, -90.0570016)
⚠️ Sin resultados para university en (13.82591739, -90.0570016)
⚠️ Sin resultados para university en (13.80091739, -90.0570016)
⚠️ Sin resultados para university en (13.77591739, -90.0570016)
⚠️ Sin resultados para university en (13.75091739, -90.0570016)
⚠️ Sin resultados para university en (13.72591739, -90.0570016)
⚠️ Sin resultados para university en (13